<a href="https://colab.research.google.com/github/mirzafani808-source/neurofive-ml-track/blob/main/customer-churn-prediction/churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -O telco_churn.csv "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

--2026-07-21 11:20:23--  https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 970457 (948K) [text/plain]
Saving to: ‘telco_churn.csv’

telco_churn.csv     100%[===================>] 947.71K  --.-KB/s    in 0.03s   

2026-07-21 11:20:23 (36.1 MB/s) - ‘telco_churn.csv’ saved [970457/970457]



In [2]:
"""
Customer Churn Prediction — Working with a Business Problem
--------------------------------------------------------------
Predicts which Telco customers are likely to churn using a
Decision Tree Classifier and Logistic Regression, and compares
their performance.
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
df = pd.read_csv("telco_churn.csv")

print("Dataset shape:", df.shape)
print(df.head())

# ---------------------------------------------------------
# 2. Quick EDA
# ---------------------------------------------------------
print("\n" + "=" * 55)
print("QUICK EDA")
print("=" * 55)

# Churn rate overall
print("\nOverall churn rate:")
print(df["Churn"].value_counts(normalize=True))

# Churn rate by contract type
print("\nChurn rate by Contract type:")
print(df.groupby("Contract")["Churn"].apply(lambda x: (x == "Yes").mean()))

# Churn rate by tenure (binned)
df["TenureGroup"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72],
                            labels=["0-12mo", "13-24mo", "25-48mo", "49-72mo"])
print("\nChurn rate by Tenure group:")
print(df.groupby("TenureGroup", observed=True)["Churn"].apply(lambda x: (x == "Yes").mean()))

# Average monthly charges: churned vs not
print("\nAverage Monthly Charges by Churn status:")
print(df.groupby("Churn")["MonthlyCharges"].mean())

# ---------------------------------------------------------
# 3. Clean data
# ---------------------------------------------------------
# TotalCharges has some blank strings for brand-new customers; convert to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Drop identifier and helper column
df = df.drop(columns=["customerID", "TenureGroup"])

# Target: convert Yes/No to 1/0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# ---------------------------------------------------------
# 4. Check class imbalance
# ---------------------------------------------------------
print("\n" + "=" * 55)
print("CLASS IMBALANCE CHECK")
print("=" * 55)
print(df["Churn"].value_counts(normalize=True))
print("Note: Churn classes are imbalanced (~73% stayed, ~27% churned).")
print("This is mentioned but not fully corrected in this task (e.g. no")
print("SMOTE/oversampling applied yet) — accuracy alone would be misleading here,")
print("so Precision/Recall/F1 are reported alongside it.\n")

# ---------------------------------------------------------
# 5. Encode categorical variables
# ---------------------------------------------------------
categorical_cols = df.select_dtypes(include="object").columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=["Churn"])
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 6. Train Logistic Regression
# ---------------------------------------------------------
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)

# ---------------------------------------------------------
# 7. Train Decision Tree
# ---------------------------------------------------------
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

# ---------------------------------------------------------
# 8. Compare performance
# ---------------------------------------------------------
def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
    }

results = pd.DataFrame([
    evaluate("Logistic Regression", y_test, log_pred),
    evaluate("Decision Tree", y_test, tree_pred),
])

print("=" * 55)
print("MODEL COMPARISON")
print("=" * 55)
print(results.to_string(index=False))

print("\nLogistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, log_pred))
print("\nDecision Tree Confusion Matrix:")
print(confusion_matrix(y_test, tree_pred))

print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, log_pred, target_names=["No Churn", "Churn"]))

print("Decision Tree Classification Report:")
print(classification_report(y_test, tree_pred, target_names=["No Churn", "Churn"]))

# ---------------------------------------------------------
# 9. Top 3 features driving churn (Decision Tree)
# ---------------------------------------------------------
importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("=" * 55)
print("TOP 5 FEATURES DRIVING CHURN (Decision Tree)")
print("=" * 55)
print(importances.head(5).to_string(index=False))


Dataset shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingM

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
